# 라벨 데이터셋 EDA

전수 라벨링이 끝난 **1,024건**을 봅니다. 요구사항 자체의 EDA는
[`03_requirements_eda.ipynb`](03_requirements_eda.ipynb)에 있고, 여기서는 **붙은 라벨**을 봅니다.

## 무엇을 확인하는가

라벨을 학습 데이터로 쓰기 전에 답해야 하는 질문 순서대로 배치했습니다.

1. **라벨이 스스로 모순되지 않는가** — 스키마에는 고정 규칙이 있습니다. `blockers`가 비어 있고
   `cost_basis`가 `없음`이면 반드시 `통상수용`이어야 합니다(결정 21, `derive_primary_action()`).
   모델 출력이 이 규칙을 어겼다면 그 행은 내부적으로 앞뒤가 안 맞는 라벨입니다.
2. **보조 축이 주 라벨과 어떻게 얽혀 있는가** — 네 축이 독립적으로 유용한지, 아니면
   주 라벨의 동어반복인지.
3. **문서별 분포가 얼마나 다른가** — 문서 단위로 학습·평가를 나누는 설계(§10.1)에서
   fold 난이도가 갈리는 정도.
4. **실행 경로가 라벨에 영향을 줬는가** — 1,024건은 동기 100 + 배치 924 혼합입니다.

## 쓰는 데이터

**확정(동결) 데이터셋** `data/labels/label_dataset_v1.jsonl`을 씁니다. 실행 결과를 그대로 합친 파일이 아니라
`build_label_dataset.py`가 정리한 것으로, 두 가지가 이미 처리돼 있습니다.

- **고정 규칙 위반 보정** — 모델 원본은 `primary_action_model`에, 보정 여부는
  `rule_corrected`에 남아 있어 아래에서 감사할 수 있습니다.
- **균일 스키마와 명시적 출처** — 모든 행이 같은 키를 갖고 `execution_path`,
  `source_run`을 직접 들고 있습니다. 조인도 추측도 필요 없습니다.

읽기는 항상 `load_label_dataset()`을 거칩니다. 로더가 SHA-256을 대조하고 값 도메인을
검사하므로, 파일이 조용히 바뀌면 분석이 도는 대신 실패합니다. 노트북 출력에 해시가 남아
**어느 시점의 라벨을 본 결과인지** 나중에 확인할 수 있습니다.

원본 실행 결과(`reports/current/claude_runs/`)는 그대로 보존돼 있습니다.
알려진 데이터 문제는 [`docs/issues/`](../docs/issues/)에 정리돼 있습니다.

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager

# notebooks/에서 열든 저장소 루트에서 열든 동작하도록 위로 거슬러 찾습니다.
ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / 'scripts').is_dir() and (p / 'data').is_dir()),
    Path.cwd().resolve(),
)

installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next(
    f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'AppleGothic', 'DejaVu Sans']
    if f in installed
)
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 40)

LABELS = ['통상수용', '견적반영', '계약·질의검토']
LEVELS = ['낮음', '보통', '높음']

# 확정 데이터셋은 로더를 통해서만 읽습니다. 로더가 SHA-256을 대조하므로
# 노트북이 어느 시점의 라벨을 봤는지가 출력에 남습니다.
sys.path.insert(0, str(ROOT))
from scripts.labeling.label_dataset import load_label_dataset

rows, meta = load_label_dataset()

df = pd.DataFrame(rows)
df['blockers'] = df['blockers'].apply(tuple)
df['blocker_count'] = df['blockers'].apply(len)
df['category'] = df['requirement_id'].str.split('-').str[0]
df['text_length'] = df['raw_requirement_text'].str.len()

print(f"데이터셋 {meta['dataset_version']} / {meta['row_count']}건 / 문서 {meta['document_count']}개")
print(f"sha256   {meta['sha256'][:16]}...")
print(f"실행 경로 {meta['execution_path_counts']}")
print(f"규칙 보정 {meta['rule_corrected_count']}건")
print(f"결측(원본 사정) {meta['nullable_missing']} — docs/issues/002")

In [ ]:
from IPython.display import Markdown, display

display(Markdown('''## 1. 라벨이 스스로 모순되지 않는가

스키마에는 **고정 규칙**이 있습니다(결정 21). 보조 축이 정해지면 주 라벨은 자동으로 결정됩니다.

```
blockers가 있으면            -> 계약·질의검토
없고 cost_basis != 없음이면  -> 견적반영
둘 다 아니면                 -> 통상수용
```

모델은 네 값을 한 번에 생성하므로 이 규칙을 어길 수 있습니다. 어긴 행은 **주 라벨과 보조 축이
서로 다른 이야기를 하는 행**이고, 학습에 넣으면 모델이 배울 수 없는 잡음이 됩니다.
'''))

from scripts.labeling.label_schema import derive_primary_action, LabelResult

# 데이터셋은 이미 보정돼 있습니다. 여기서는 (1) 보정이 실제로 규칙과 맞는지,
# (2) 보정 후 남은 위반이 없는지를 감사합니다.
def rule_label(row):
    if row['blocker_count'] > 0:
        return '계약·질의검토'
    return '통상수용' if row['cost_basis'] == '없음' else '견적반영'

df['rule_action'] = df.apply(rule_label, axis=1)
assert (df['rule_action'] == df['primary_action']).all(), '보정 후에도 규칙 위반이 남아 있습니다'

corrected = df[df['rule_corrected']]
dist = df['primary_action'].value_counts().reindex(LABELS)
model_dist = df['primary_action_model'].value_counts().reindex(LABELS)
display(pd.DataFrame({
    '모델 원본': model_dist,
    '보정 후': dist,
    '차이': dist - model_dist,
    '비율(%)': (dist / len(df) * 100).round(1),
}))

print(f'규칙 보정 {len(corrected)}건 / {len(df)}건 ({len(corrected)/len(df):.1%})')
display(
    corrected[['requirement_uid', 'primary_action_model', 'primary_action',
               'blockers', 'cost_basis', 'reasoning']]
    .rename(columns={'primary_action_model': '모델 출력', 'primary_action': '규칙 적용'})
    .set_index('requirement_uid')
)
print('reasoning을 읽으면 6건 모두 근거 문장이 규칙 쪽을 지지합니다.')
print('특히 koen_ai_infrastructure:CON-004는 모델이 reasoning 안에서 스스로 정정해놓고')
print('필드는 고치지 않은 경우입니다. 보정은 임의 덮어쓰기가 아니라 자기 근거와 맞추는 일입니다.')

In [ ]:
display(Markdown('''## 2. 보조 축은 독립적으로 쓸모가 있는가

보조 축이 주 라벨의 동어반복이면 축을 나눈 의미가 없습니다. `blockers`와 `cost_basis`는
정의상 주 라벨을 결정하므로 당연히 얽혀 있고, 실제로 봐야 할 것은 **`domain_dependency`와
`build_difficulty`가 주 라벨과 얼마나 독립인가**입니다. 이 둘은 규칙에 들어가지 않습니다.

프롬프트에서 두 축을 분리하고 "난이도가 높다는 사실만으로 주 라벨을 올리지 않는다"고
명시한 것(결정 20)이 실제로 지켜졌는지를 여기서 확인합니다.
'''))

blocker_counts = Counter(b for bs in df['blockers'] for b in bs)
pair_counts = Counter(
    pair for bs in df['blockers'] for pair in combinations(bs, 2)
)

print(f'blocker 없음 {(df["blocker_count"] == 0).sum()}건 '
      f'({(df["blocker_count"] == 0).mean():.1%}) / '
      f'최대 {df["blocker_count"].max()}개 동시 부여')
display(pd.DataFrame(blocker_counts.most_common(), columns=['blocker', '건수']).set_index('blocker'))
display(pd.DataFrame(
    [(' + '.join(p), n) for p, n in pair_counts.most_common(5)],
    columns=['동시 부여 조합', '건수'],
).set_index('동시 부여 조합'))

display(pd.crosstab(df['cost_basis'], df['primary_action'])
        .reindex(columns=LABELS, fill_value=0)
        .sort_values('견적반영', ascending=False))

# 난이도 축과 주 라벨의 관계. 행 정규화해서 각 난이도에서의 라벨 비율을 봅니다.
for axis in ('domain_dependency', 'build_difficulty'):
    tab = pd.crosstab(df[axis], df['primary_action'], normalize='index')
    tab = (tab.reindex(index=LEVELS, columns=LABELS) * 100).round(1)
    display(tab.style.set_caption(f'{axis} 수준별 주 라벨 비율 (%)'))

cramer = pd.crosstab(df['domain_dependency'], df['build_difficulty'])
display(cramer.reindex(index=LEVELS, columns=LEVELS)
        .style.set_caption('두 난이도 축의 교차 (독립적으로 판단됐는지)'))

In [ ]:
display(Markdown('''## 3. 문서별 분포 — fold 난이도가 얼마나 갈리는가

문서 단위로 학습·평가를 나눌 계획입니다(§10.1). 같은 요구사항이 여러 RFP에 반복되므로
무작위 분할은 누수가 되기 때문입니다.

대신 **문서마다 라벨 분포가 다르면 fold마다 난이도가 달라집니다.** 통상수용이 80%인 문서를
평가에 쓰면 다수 클래스만 찍어도 점수가 높게 나오고, 30%인 문서를 쓰면 같은 모델이
훨씬 낮게 나옵니다. 성능 차이가 모델이 아니라 분할에서 오는 셈입니다.
'''))

doc_stats = (
    df.groupby('document_id')
      .agg(건수=('requirement_uid', 'size'),
           통상수용비율=('primary_action', lambda s: (s == '통상수용').mean()),
           blocker보유율=('blocker_count', lambda s: (s > 0).mean()),
           평균본문길이=('text_length', 'mean'))
      .sort_values('통상수용비율')
)
doc_stats['통상수용비율'] = (doc_stats['통상수용비율'] * 100).round(1)
doc_stats['blocker보유율'] = (doc_stats['blocker보유율'] * 100).round(1)
doc_stats['평균본문길이'] = doc_stats['평균본문길이'].round(0).astype(int)
display(doc_stats)

lo, hi = doc_stats['통상수용비율'].iloc[0], doc_stats['통상수용비율'].iloc[-1]
print(f'통상수용 비율 {lo}% ~ {hi}% (최대/최소 {hi/lo:.1f}배)')
print('문서를 그대로 fold로 쓰면 이 편차가 그대로 fold 난이도 편차가 됩니다.')
print('층화 분할이나 fold별 기준선(다수 클래스 정확도) 보고가 필요합니다.\n')

# 다수 클래스만 찍는 기준선. fold별로 이 값이 다르면 절대 점수 비교가 무의미합니다.
baseline = df.groupby('document_id')['primary_action'].apply(
    lambda s: s.value_counts(normalize=True).max()
).sort_values()
print('문서별 다수 클래스 기준선(이 점수 아래면 모델이 무의미):')
print((baseline * 100).round(1).to_string())

# 요구사항 카테고리별 분포. 접두어가 요구사항 유형을 나타냅니다.
top_categories = df['category'].value_counts().head(8).index
cat_tab = pd.crosstab(
    df[df['category'].isin(top_categories)]['category'],
    df[df['category'].isin(top_categories)]['primary_action'],
    normalize='index',
).reindex(columns=LABELS)
display((cat_tab * 100).round(1).style.set_caption('주요 요구사항 카테고리별 주 라벨 비율 (%)'))

In [ ]:
display(Markdown('''## 4. 실행 경로가 라벨에 영향을 줬는가 — 측정 불가

1,024건은 동기 100 + 배치 924 혼합입니다(Chunk 1 재실행 배치를 취소했기 때문).
경로별 분포를 그냥 비교하면 큰 차이가 보이지만, **그 차이를 실행 경로 탓으로 돌릴 수 없습니다.**

동기로 돌린 100건 중 95건이 한 문서(`ccrs_ai_platform`)에 몰려 있고, 그 문서는 다른 경로로
돌린 건이 하나도 없습니다. 즉 **실행 경로와 문서가 완전히 교란(confounded)되어 있어**
둘을 분리할 수 없습니다. 아래에서 그 사실 자체를 확인합니다.
'''))

path_tab = pd.crosstab(df['execution_path'], df['primary_action'], normalize='index')
display((path_tab.reindex(columns=LABELS) * 100).round(1)
        .style.set_caption('실행 경로별 주 라벨 비율 (%) — 해석하면 안 되는 표'))

overlap = (
    df.groupby('document_id')['execution_path'].nunique()
      .pipe(lambda s: s[s > 1]).index.tolist()
)
print('두 경로가 모두 존재하는 문서:', overlap or '없음')
for doc in overlap:
    sub = df[df['document_id'] == doc]
    tab = pd.crosstab(sub['execution_path'], sub['primary_action'], normalize='index')
    counts = sub['execution_path'].value_counts().to_dict()
    print(f'\n{doc} {counts}')
    display((tab.reindex(columns=LABELS) * 100).round(1))
print('\n비교 가능한 표본이 이 정도면 경로 효과를 추정할 수 없습니다.')
print('결론: 실행 경로의 영향은 이 데이터로 측정 불가입니다. 측정하려면 같은 문서를')
print('양쪽 경로로 돌린 표본이 필요하고, 그것이 결정 27에서 시도했다가 취소한 작업입니다.\n')

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

df['primary_action'].value_counts().reindex(LABELS).plot.bar(
    ax=axes[0], rot=15, color=['#4C78A8', '#F58518', '#E45756'])
axes[0].set(title=f'주 라벨 분포 (n={len(df)})', ylabel='건수', xlabel='')

doc_stats['통상수용비율'].plot.barh(ax=axes[1], color='#54A24B')
axes[1].axvline(df['primary_action'].eq('통상수용').mean() * 100,
                color='crimson', linestyle='--', label='전체 평균')
axes[1].set(title='문서별 통상수용 비율 (%)', xlabel='%', ylabel='')
axes[1].legend()

pd.Series(blocker_counts).sort_values().plot.barh(ax=axes[2], color='#B279A2')
axes[2].set(title='blocker 유형별 부여 건수', xlabel='건수', ylabel='')

plt.tight_layout()

print('=' * 62)
print('요약')
print(f'  - 라벨 {len(df)}건, 규칙 보정 {len(corrected)}건 '
      f'({len(corrected)/len(df):.1%}) — 모델 원본은 primary_action_model에 보존')
print(f'  - blocker 미부여 {(df["blocker_count"] == 0).mean():.1%}, '
      f'추가 원가 없음 {(df["cost_basis"] == "없음").mean():.1%}')
print(f'  - 문서별 통상수용 {lo}% ~ {hi}% — fold별 기준선을 반드시 함께 보고')
print('  - 실행 경로 효과: 문서와 교란되어 측정 불가')